# Lab 3: Local RAG System with Source Attribution and Scope Detection

**Duration:** ~20 minutes

---

Labs 1 and 2 showed how to explain ML models that classify structured data. Now we move to **LLMs**: large language models that process and generate unstructured (and mainly text) data.

A **Retrieval-Augmented Generation (RAG)** system is one of the most important transparency patterns for LLMs: instead of relying on what the model "memorized" during training, you make it ground its answers in retrieved custom documents. This improves trustworthiness, because you can point to where (which document) the model retrieved its information from.

In this lab you will **build and evaluate source attribution, inline citations, and scope detection on top of a very simple RAG system**. The focus is not on RAG itself (this is why we won't be building embeddings nor complex Goodness-of-Fit retrieval metrics), but instead on the mentioned transparency capabilities. During this lab, you will first:

1. Load a small knowledge base of `.txt` files
2. Retrieve relevant documents using **naive keyword search** (no embeddings, no vectors)
3. Generate answers via a **local LLM** (Ollama) grounded in the retrieved context
4. Add **source attribution**, by showing which documents were used
5. Run the full pipeline on several queries and observe where it works and where it fails

---

## Initial architecture

```
User query
    │
    ▼
┌─────────────────────┐
│  Retrieval Engine   │  ← naive keyword matching against .txt files
│  (keyword search)   │    returns top-K documents by overlap score
└─────────────────────┘
    │ top-k documents + titles (as metadata)
    ▼
┌─────────────────────┐
│  Prompt Builder     │  ← injects retrieved context into the prompt
│  (with attribution) │    includes document titles as sources
└─────────────────────┘
    │ grounded prompt
    ▼
┌─────────────────────┐
│  Answer Generation  │  ← local LLM via Ollama
│  (local LLM)        │
└─────────────────────┘
```

---

## Prerequisites

- [Ollama](https://ollama.com) running locally with a small model pulled (`qwen3:0.6b`)
- The `knowledge/` folder in the same directory as this notebook, containing `.txt` documents
- Python packages: `requests` (standard in most environments)

## Step 1: Imports and Configuration

We only need the Python standard library (`os`) and `requests` to call the Ollama API. No embeddings nor ML frameworks: the point is to show the **core RAG concept** first, with human-readable documents and a human-understandable retrieval mechanism (simple keyword matching excluding very short words -less than 4 characters-).

In [23]:
import os
import requests

# --- Ollama configuration ---
OLLAMA_URL = "http://localhost:11434/api/chat"
OLLAMA_MODEL = "qwen3:0.6b"  # adjust to match your local Ollama model name

# --- Retrieval configuration ---
TOP_K = 2          # number of documents to retrieve per query
MIN_WORD_LEN = 4   # ignore short words (stop-word filtering heuristic)

print("Configuration loaded.")
print(f"  Ollama model : {OLLAMA_MODEL}")
print(f"  Top-K docs   : {TOP_K}")

Configuration loaded.
  Ollama model : qwen3:0.6b
  Top-K docs   : 2


## Step 2: Load the Knowledge Base

The knowledge base is a folder of plain `.txt` files. Each file follows a simple convention:
- **Line 1**: document title
- **Remaining lines**: document body

We load every `.txt` file into a Python dictionary: `{ title → body text }`.

For this lab, let's run an imagination exercise: due to the rapid AI advancements in the last years, we are now living in a parallel universe where Europe has become a dystopian, cyberpunk world. Capitals like Berlin are massively overpopulated, due to their popular industrial and technified atmosphere. Climate change has turned Northern countries into mild, Mediterranean-like climates. AI has optimized nuclear fusion, and now almost everyone moves around in fusion-powered flying cars. Everyone has quantum computers at home.

We have built a simple RAG knowledge base (under the `knowledge/` folder) with every major change that this futuristic Europe has undergone during 2026. LLMs have of course been trained on now outdated information (pre-our 2026) about European countries, so we want to ground their answers on this database.

Feel free to open the files in `knowledge/` and read their content before running the following cell.

In [24]:
def load_knowledge(folder):
    """Load all .txt files from `folder` into a {title: body} dict."""
    knowledge = {}
    for filename in sorted(os.listdir(folder)):
        if filename.endswith(".txt"):
            filepath = os.path.join(folder, filename)
            with open(filepath, "r", encoding="utf-8") as f:
                lines = f.read().strip().splitlines()
                title = lines[0]
                body = "\n".join(lines[1:]).strip()
                knowledge[title] = body
    return knowledge


KNOWLEDGE_FOLDER = os.path.join(os.path.dirname(os.path.abspath("__file__")), "knowledge")
knowledge = load_knowledge(KNOWLEDGE_FOLDER)

print(f"Loaded {len(knowledge)} documents from '{KNOWLEDGE_FOLDER}':\n")
for title, body in knowledge.items():
    preview = body[:80].replace("\n", " ")
    print(f"  [{title}]\n   {preview}...\n")

Loaded 6 documents from 'c:\Users\luciac\Documents\XAI-for-LLMs-Labs\lab3-rag-scope-detector\knowledge':

  [Germany in 2026]
   The population of Berlin has almost doubled during 2026, going up to 7 million i...

  [Netherlands in 2026]
   Due to international pressure and the high language level of its citizens, the N...

  [Norway in 2026]
   Because of climate change, Norway's weather in 2026 is now similar to Mediterran...

  [Quantum computing in 2026]
   Quantum computers have become the standard end-user technology for Europeans in ...

  [Spain in 2026]
   As of 2026, the new capital of Spain is Barcelona, replacing Madrid which has be...

  [Transportation in Europe in 2026]
   The most used means of transport in all countries of the European Union is the f...



## Step 3: Retrieval with Naive Keyword Search

We use the simplest possible retrieval strategy: **count how many query words appear in each document** and return the top-K documents by overlap score.

- We filter out short words (length < `MIN_WORD_LEN`) as a basic stop-word filter.
- We only return documents that have **at least one matching word** (score > 0).

This approach has obvious weaknesses (no synonyms, no semantic similarity); we will discuss them in the exercises.

In [25]:
def naive_keyword_search(query, documents, top_k=TOP_K):
    """
    Score each document by counting keyword overlaps with the query.
    Returns a list of (title, body) tuples for the top_k matches.
    """
    query_words = set(
        w for w in query.lower().split() if len(w) >= MIN_WORD_LEN
    )

    scored = []
    for title, body in documents.items():
        doc_words = set(
            w for w in body.lower().split() if len(w) >= MIN_WORD_LEN
        )
        score = len(query_words & doc_words)  # intersection size
        scored.append({"title": title, "body": body, "score": score})

    scored.sort(key=lambda x: x["score"], reverse=True)

    return [
        (item["title"], item["body"])
        for item in scored[:top_k]
        if item["score"] > 0
    ]


# Quick smoke test
test_query = "What languages do Dutch people speak?"
results = naive_keyword_search(test_query, knowledge)
print(f"Query: \"{test_query}\"")
print(f"Retrieved {len(results)} document(s):")
for title, _ in results:
    print(f"  - {title}")

Query: "What languages do Dutch people speak?"
Retrieved 1 document(s):
  - Netherlands in 2026


## Step 4: Answer Generation with Simple Source Attribution (Document Titles)

We call a **local LLM via Ollama** and inject the retrieved documents into the prompt as context. Each context snippet is labelled with its **document title**; this metadata is what will give us the attribution.

In [26]:
def generate_answer(query, retrieved_docs):
    """
    Call the local LLM with the query and retrieved context.

    Parameters
    ----------
    query          : str              — the user's question
    retrieved_docs : list[(str, str)] — list of (title, body) pairs

    Returns
    -------
    answer  : str        — the LLM's response
    sources : list[str]  — document titles used as context
    """

    sources = [title for title, _ in retrieved_docs]

    # Build context block with labelled source snippets
    context_block = "\n\n".join(
        f"[Source: {title}]\n{body}"
        for title, body in retrieved_docs
    )

    prompt = (
        f"Context: {context_block}\n\n"
        f"Question: {query}"
    )

    response = requests.post(
        OLLAMA_URL,
        json={
            "model": OLLAMA_MODEL,
            "messages": [{"role": "user", "content": prompt}],
            "stream": False,
        },
        timeout=60,
    )
    response.raise_for_status()

    answer = response.json()["message"]["content"]
    return answer, sources

We combine retrieval and generation into a single `rag_pipeline` function that:

1. Retrieves the most relevant documents for a query
2. Generates an answer via the local LLM
3. Prints the **sources** (document titles) alongside the answer

In [27]:
def rag_pipeline(query):
    """Run the full RAG pipeline: retrieve → generate → display with attribution."""
    print(f"\n{'='*60}")
    print(f"Question: {query}")
    print('='*60)

    # Step 1: Retrieve
    retrieved = naive_keyword_search(query, knowledge)
    if retrieved:
        print(f"\nRetrieved {len(retrieved)} document(s):")
        for title, _ in retrieved:
            print(f"  - {title}")
    else:
        print("\nNo relevant documents found in the knowledge base.")

    # Step 2: Generate
    answer, sources = generate_answer(query, retrieved)

    # Step 3: Display with attribution
    print(f"\nAnswer:\n{answer}")
    if sources:
        print(f"\nSources: {', '.join(sources)}")
    print()

Let's run the pipeline on a set of example queries. As you read the output, consider:

- Does the system retrieve the **correct** document(s)?
- Is the answer **grounded** in the retrieved text, or does the LLM add information from its training data?
- Does the attribution (source titles) help you **trust** the answer more?
- What happens when the query is **outside** the knowledge base?

In [15]:
rag_pipeline("What is the current population of Berlin?")


Question: What is the current population of Berlin?

Retrieved 1 document(s):
  - Germany in 2026

Answer:
The current population of Berlin is 7 million inhabitants as stated in the 2026 data.

Sources: Germany in 2026



In [16]:
rag_pipeline("What's the weather like in Norway during February?")


Question: What's the weather like in Norway during February?

Retrieved 2 document(s):
  - Germany in 2026
  - Norway in 2026

Answer:
In Norway in 2026, February is expected to have a **Mediterranean climate**, characterized by:  
- **Warm temperatures** in the summer (May to August)  
- **Cool temperatures** in the winter (February to March)  
- **Rainy and dry conditions** throughout the year.  

This aligns with the general Mediterranean climate pattern, where winters are cold, wet, and dry, while summers are warm and humid.

Sources: Germany in 2026, Norway in 2026



In [8]:
rag_pipeline("What languages do Dutch people speak?")


Question: What languages do Dutch people speak?

Retrieved 1 document(s):
  - Netherlands in 2026

Answer:
Dutch people still speak Dutch as their official language. Additionally, the Netherlands still maintains Frisian as its official language. However, it's important to note that the first official language in the Netherlands was English, which was implemented in July 2026.

Sources: Netherlands in 2026



In [9]:
rag_pipeline("Tell me about quantum computing")


Question: Tell me about quantum computing

Retrieved 1 document(s):
  - Quantum computing in 2026

Answer:
In 2026, quantum computing is recognized as a cornerstone of European technological innovation, marking a pivotal shift in how European industries approach advanced technologies. Here's a structured overview:

1. **Standardization**: Quantum computing is positioned as a standard end-user technology, emphasizing its integration into everyday European applications. This includes sectors like healthcare, finance, and communication, where quantum algorithms are expected to revolutionize processes.

2. **European Context**: The EU’s focus on quantum computing underscores its role in driving global innovation. Initiatives such as the European Quantum Computing Program aim to foster collaboration and funding to accelerate quantum research and development.

3. **Applications**: Quantum computing’s potential lies in fields such as cryptography, material science, and optimization. For exam

In [10]:
rag_pipeline("What is the capital of Spain?")


Question: What is the capital of Spain?

Retrieved 1 document(s):
  - Spain in 2026

Answer:
As of 2026, the capital of Spain is **Barcelona**.

Sources: Spain in 2026



We know that our provided RAG database contains information about Europe from 2026 on. But what if we ask the LLM about something totally unrelated?:

In [11]:
# This query is completely outside the knowledge base. What happens when we run it?
rag_pipeline("How can I install Python on my Windows laptop?")


Question: How can I install Python on my Windows laptop?

No relevant documents found in the knowledge base.

Answer:
To install Python on your Windows laptop, follow these steps:

1. **Download Python**:  
   Visit the official [Python website](https://www.python.org/) and download the installer. Choose the version compatible with your operating system (e.g., 32-bit or 64-bit depending on your laptop).

2. **Run the Installer**:  
   Once the .exe file is downloaded, double-click it to install Python.  
   - **Windows 7/8/10**: Ensure your system has the correct drivers installed (e.g., graphics or chipset drivers) to ensure compatibility.  
   - **64-bit systems**: If your laptop is 64-bit, install Python in 64-bit mode.

3. **Verify Installation**:  
   After installation, open a command prompt (type `python --version`) or a PowerShell shell to confirm Python is installed. If it’s installed correctly, it should display the correct version number.

### Additional Notes:
- Python 3 i

Even if no relevant documents were retrieved, the LLM still generated an answer based on its training data. This training data might be reliable. Or not. We cannot know because it is not accessible anymore, so we cannot validate it.

We want to avoid these situations. For now, we can simply skip doing an LLM call if no relevant documents were retrieved from the RAG database. This not only gives us more control over the system and more trust, but we are also avoiding unnecessary LLM calls which, in the long term, can significantly reduce costs money- and time-wise. Let's simply add an extra if statement in our generate_answer method:

In [28]:
def generate_answer(query, retrieved_docs):
    """
    Call the local LLM with the query and retrieved context.

    Parameters
    ----------
    query          : str              — the user's question
    retrieved_docs : list[(str, str)] — list of (title, body) pairs

    Returns
    -------
    answer  : str        — the LLM's response
    sources : list[str]  — document titles used as context
    """
    if not retrieved_docs: # NEW CODE BLOCK
        return (
            "I don't have relevant information in my knowledge base to answer that question.",
            []
        )

    sources = [title for title, _ in retrieved_docs]

    # Build context block with labelled source snippets
    context_block = "\n\n".join(
        f"[Source: {title}]\n{body}"
        for title, body in retrieved_docs
    )

    prompt = (
        f"Context: {context_block}\n\n"
        f"Question: {query}"
    )

    response = requests.post(
        OLLAMA_URL,
        json={
            "model": OLLAMA_MODEL,
            "messages": [{"role": "user", "content": prompt}],
            "stream": False,
        },
        timeout=60,
    )
    response.raise_for_status()

    answer = response.json()["message"]["content"]
    return answer, sources

Now, if we run the same query, the LLM shouldn't generate any answer:

In [20]:
# This query is completely outside the knowledge base.
# Our guardrail blocks the LLM call and returns a safe static message instead.
rag_pipeline("How can I install Python on my Windows laptop?")


Question: How can I install Python on my Windows laptop?

No relevant documents found in the knowledge base.

Answer:
I don't have relevant information in my knowledge base to answer that question.



## Step 7: Inspect the Retrieval Scores

It helps to see the **scores** for every document, not just the winners. This makes the retrieval decision transparent and lets you spot cases where the wrong document is ranked first.

In [29]:
def explain_retrieval(query, documents):
    """Print the keyword overlap score for every document, ranked."""
    query_words = set(
        w for w in query.lower().split() if len(w) >= MIN_WORD_LEN
    )
    print(f"Query words (len >= {MIN_WORD_LEN}): {query_words}\n")

    scored = []
    for title, body in documents.items():
        doc_words = set(
            w for w in body.lower().split() if len(w) >= MIN_WORD_LEN
        )
        matched = query_words & doc_words
        scored.append((title, len(matched), matched))

    scored.sort(key=lambda x: x[1], reverse=True)

    print(f"{'Document title':<40} {'Score':>5}  Matched words")
    print("-" * 75)
    for title, score, matched in scored:
        matched_str = ", ".join(sorted(matched)) if matched else "-"
        print(f"  {title:<38} {score:>5}  {matched_str}")


explain_retrieval("What's the weather like in Norway during February?", knowledge)

Query words (len >= 4): {'like', 'norway', 'during', 'february?', "what's", 'weather'}

Document title                           Score  Matched words
---------------------------------------------------------------------------
  Germany in 2026                            2  during, weather
  Norway in 2026                             1  weather
  Netherlands in 2026                        0  -
  Quantum computing in 2026                  0  -
  Spain in 2026                              0  -
  Transportation in Europe in 2026           0  -


In the example query about Norway's weather, we see that the wrong document (`Germany in 2026`) was ranked before the correct document, due to a higher retrieval score (more keywords matched).

We of course know that this is a naive retrieval mechanism. But hopefully, this will show you that we shouldn't blindly trust any given attributed sources, and we should look into those source documents ourselves.

Also, source attribution based solely on metadata (the document titles) doesn't give us a lot of trust. Let's check this query again:

In [30]:
rag_pipeline("Tell me about quantum computing")


Question: Tell me about quantum computing

Retrieved 1 document(s):
  - Quantum computing in 2026

Answer:
Quantum computing is set to become a mainstream technology in 2026, with Europe leading the way in its development and application. Here's a breakdown of key aspects:

1. **Advancements in Quantum Hardware**:  
   Significant progress has been made in quantum processors. The Shor's algorithm, a breakthrough in 2023, enables quantum computers to factor large numbers exponentially faster than classical methods. This has redefined computational capabilities, particularly in fields like cryptography, where quantum algorithms could break traditional encryption methods.

2. **Quantum Supremacy**:  
   By 2026, quantum computers are expected to surpass classical machines in solving specific problems, such as complex mathematical simulations or optimizing processes like drug discovery. This shift could redefine industries reliant on these capabilities.

3. **European Initiatives**:  
   

A correct document is retrieved, and the LLM answer does mention that Europe has widely adopted quantum computing.

However, most of the generated information does surely not come from the retrieved document. Instead, the LLM is generating its answer based on its training data.

We know from the Python-related query that we cannot blindly trust these non-RAG-grounded answers. The problem here is that we are given a source attribution to a seemingly reliable document from our knowledge base. This might make us trust the answer more, but it shouldn': we don't know what part of the answer is grounded in our sourced document, and what isn't.

Inline citations can help us with this.

In [ ]:
# TODO: continue Lab by extending the answering function:

def generate_answer(query, retrieved_docs):
    """
    Call the local LLM with the query and retrieved context.

    Parameters
    ----------
    query          : str              — the user's question
    retrieved_docs : list[(str, str)] — list of (title, body) pairs

    Returns
    -------
    answer  : str        — the LLM's response
    sources : list[str]  — document titles used as context
    """
    if not retrieved_docs:
        return (
            "I don't have relevant information in my knowledge base to answer that question.",
            []
        )

    sources = [title for title, _ in retrieved_docs]

    # Build context block with labelled source snippets
    context_block = "\n\n".join(
        f"[Source: {title}]\n{body}"
        for title, body in retrieved_docs
    )

    prompt = (
        "Use only the context below to answer the question. "
        "If the answer is not in the context, say you don't know.\n\n"
        f"{context_block}\n\n"
        f"Question: {query}"
    )

    response = requests.post(
        OLLAMA_URL,
        json={
            "model": OLLAMA_MODEL,
            "messages": [{"role": "user", "content": prompt}],
            "stream": False,
        },
        timeout=60,
    )
    response.raise_for_status()

    answer = response.json()["message"]["content"]
    return answer, sources

## Step 8: Inline Citations

Until now, we have implemented simple source attribution: showing document titles at the end of an answer. This tells us *which* documents were in the context, but not *which part of the answer* each document is responsible for. A grounded claim and a hallucinated claim would look identical to us if they both display the same retrieved document titles.

**Inline citations** fix this by attaching a `[source]` marker at the end of sentences that are supported by that source.

We attribute sentences to sources by measuring what disappears when a source is removed:

---

```
Step 0   Call the LLM with ALL retrieved documents → full answer (baseline)

Step 1   Remove document [1] → call LLM again → answer_without_1
Step 2   Remove document [2] → call LLM again → answer_without_2
         … repeat for every retrieved document …

Step 3   Split the full answer into sentences.
         For each sentence, for each removed document [i]:
           ratio = (words in sentence absent from answer_without_i) / (words in sentence)
           if ratio ≥ CITATION_THRESHOLD  →  attribute sentence to [i]
         Append [i] citation markers at the end of attributed sentences.
```

**Cost:** `1 + n_docs` LLM calls per query. With `TOP_K = 2` that is 3 calls total.

---

**Key caveats and limitations to keep in mind:**
- We use `temperature=0` to make the LLM as deterministic as possible across calls, but with how LLMs are implemented, they can never be fully deterministic. Small variations may still occur.
- A sentence marked `[i]` means *"removing document i caused a significant fraction of this sentence's words to disappear"*, which is a correlation, but not a guaranteed proof of faithful grounding.
- Sentences with no citations are likely drawn from the model's training data.
- The LLM might express the same fact using different words (paraphrasing) in the LLM calls with removed documents; the original words will then appear uncited even though they're grounded.
- Function words ("the", "is", "in") may appear in all answers regardless of context, so they're never attributed to a specific document even if they might be grounded on it.
- `CITATION_THRESHOLD` controls sensitivity: lower values attribute more sentences (higher recall), higher values are more conservative (higher precision).

In [ ]:
import re

# Fraction of a sentence's words that must disappear (when a source is removed)
# for that sentence to be attributed to that source. Range: 0.0 – 1.0.
CITATION_THRESHOLD = 0.6


def _strip_think_tags(text):
    """Remove <think>...</think> reasoning blocks produced by some models (e.g. qwen3)."""
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()


def _call_llm_grounded(query, context_block):
    """
    Single LLM call with a strict grounding instruction.
    Uses temperature=0 for deterministic (reproducible) output across LOO (leave-one-out) calls.
    Returns an empty string if context_block is empty.
    """
    
    prompt = (
        f"Context: {context_block}\n\n"
        f"Question: {query}"
    )
    response = requests.post(
        OLLAMA_URL,
        json={
            "model": OLLAMA_MODEL,
            "messages": [{"role": "user", "content": prompt}],
            "stream": False,
            "options": {"temperature": 0},
        },
        timeout=60,
    )
    response.raise_for_status()
    raw = response.json()["message"]["content"]
    return _strip_think_tags(raw)

print(f"  Citation threshold : {CITATION_THRESHOLD}")

  Citation threshold : 0.7


In [56]:
def generate_answer_with_inline_citations(query, retrieved_docs):
    """
    Sentence-level inline citation algorithm.

    Algorithm
    ---------
    0. Call the LLM with ALL retrieved documents → full_answer (baseline)
    1. For each document i, remove it and call the LLM again → answer_without_i
    2. Split full_answer into sentences. For each sentence, compute:
         ratio = (words in sentence absent from answer_without_i) / (words in sentence)
       If ratio >= CITATION_THRESHOLD, the sentence is attributed to document [i].
    3. Append [i] citation markers at the end of attributed sentences.

    Parameters
    ----------
    query          : str              — the user's question
    retrieved_docs : list[(str, str)] — list of (title, body) pairs

    Returns
    -------
    annotated_answer : str       — full answer with sentence-level [i] citation markers
    sources          : list[str] — document titles (1-indexed, matching citations)
    """
    if not retrieved_docs:
        return (
            "I don't have relevant information in my knowledge base to answer that question.",
            []
        )
    
    sources = [title for title, _ in retrieved_docs]

    def build_context(docs):
        return "\n\n".join(
            f"[Source: {title}]\n{body}" for title, body in docs
        )

    def normalise(word):
        """Strip punctuation and lowercase a word for set-based comparison."""
        return re.sub(r"[^\w]", "", word).lower()

    # Step 0: baseline answer with all documents
    print("  Calling LLM with all documents (baseline)...")
    full_answer = _call_llm_grounded(query, build_context(retrieved_docs))

    # Steps 1-2: one LOO call per retrieved document
    loo_word_sets = []
    for i, (title, _) in enumerate(retrieved_docs):
        docs_without_i = [d for j, d in enumerate(retrieved_docs) if j != i]
        print(f"  Calling LLM without document [{i + 1}] ({title})...")
        ans_without_i = _call_llm_grounded(query, build_context(docs_without_i))
        loo_word_sets.append(
            {normalise(w) for w in ans_without_i.split() if normalise(w)}
        )

    # Step 3: sentence-level attribution
    # Split on whitespace that follows sentence-ending punctuation (.  !  ?)
    sentences = re.split(r'(?<=[.!?])\s+', full_answer.strip())

    annotated_sentences = []
    for sentence in sentences:
        words = [normalise(w) for w in sentence.split() if normalise(w)]
        if not words:
            annotated_sentences.append(sentence)
            continue

        citing = []
        for i, word_set in enumerate(loo_word_sets):
            words_gone = sum(1 for w in words if w not in word_set)
            ratio = words_gone / len(words)
            if ratio >= CITATION_THRESHOLD:
                citing.append(str(i + 1))

        if citing:
            annotated_sentences.append(sentence + "".join(f"[{c}]" for c in citing))
        else:
            annotated_sentences.append(sentence)

    annotated_answer = " ".join(annotated_sentences)
    return annotated_answer, sources

In [57]:
def rag_pipeline_with_citations(query):
    """RAG pipeline with leave-one-out inline citation attribution."""
    print(f"\n{'='*60}")
    print(f"Question: {query}")
    print('='*60)

    retrieved = naive_keyword_search(query, knowledge)
    if retrieved:
        print(f"\nRetrieved {len(retrieved)} document(s):")
        for i, (title, _) in enumerate(retrieved, 1):
            print(f"  [{i}] {title}")
    else:
        print("\nNo relevant documents found in the knowledge base.")

    print()
    answer, sources = generate_answer_with_inline_citations(query, retrieved)

    print(f"\nAnswer (with inline citations):\n{answer}")
    if sources:
        print("\nSources:")
        for i, title in enumerate(sources, 1):
            print(f"  [{i}] {title}")
    print()

Let's run the same queries as before and compare the output. Notice how the answer now shows `[1]` or `[2]` markers inline, right after words that are attributed to the corresponding source document.

In [58]:
rag_pipeline_with_citations("Tell me about quantum computing")


Question: Tell me about quantum computing

Retrieved 1 document(s):
  [1] Quantum computing in 2026

  Calling LLM with all documents (baseline)...
  Calling LLM without document [1] (Quantum computing in 2026)...

Answer (with inline citations):
Quantum computing, which leverages quantum bits (qubits) to perform calculations in superposition and entanglement, has emerged as a transformative technology. By enabling exponential speedups over classical computers, quantum systems could solve complex problems far more efficiently than traditional machines. In 2026, this field has become a standard end-user technology for Europeans, signaling a pivotal shift in global innovation. **Applications of Quantum Computing:**  
- **Cryptography:** Quantum computers could break traditional encryption methods, revolutionizing secure communication. - **Drug Discovery:** By simulating molecular interactions, quantum systems could accelerate drug development, reducing time and cost. - **Materials Scien

In [52]:
rag_pipeline_with_citations("What's the weather like in Norway during February?")


Question: What's the weather like in Norway during February?

Retrieved 2 document(s):
  [1] Germany in 2026
  [2] Norway in 2026

  Calling LLM with all documents (baseline)...
  Calling LLM without document [1] (Germany in 2026)...
  Calling LLM without document [2] (Norway in 2026)...

Answer (with inline citations):
In Norway during February 2026, the weather is expected to be similar to Mediterranean climates throughout the year. This means the temperature will be warm and dry, with less precipitation, as climate change has influenced the region's weather patterns.[2]

Sources:
  [1] Germany in 2026
  [2] Norway in 2026



Look at the annotated answers and ask yourself the following questions:

1. Which sentences carry citation markers, and which don't? Do the un-cited sentences seem less trustworthy to you now (like generic filler or hallucinated content)?

2. Pick a word in a cited sentence and check it against the source document listed. Is the word actually in that document? Is the attribution to that document accurate? Does the surrounding sentence in the answer reflect the document's content faithfully?

3. Run the same query twice. Do you get identical citation patterns? Why or why not?

4. Modify the CITATION_THRESHOLD and re-run the queries. How do the inline citations change? How do these changes affect your own trust in the LLM answer? Is there a threshold where you seem to obtain more accurate citations?